In [13]:
import pandas as pd
import numpy as np
import math
import unittest
import io

from sqlalchemy import create_engine, Column, Float, Integer, String, text
from sqlalchemy.orm import declarative_base, Session

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, Legend, HoverTool
from bokeh.layouts import gridplot
from bokeh.palettes import Category10

output_notebook()  # Makes Bokeh render inside Colab

print("✅ All imports successful")


✅ All imports successful


In [3]:
from google.colab import files

print("📂 Upload your CSV files when prompted...")
print("   Upload: train.csv, test.csv, ideal.csv")

uploaded = files.upload()

# Load into DataFrames
train_df  = pd.read_csv(io.BytesIO(uploaded['train.csv']))
test_df   = pd.read_csv(io.BytesIO(uploaded['test.csv']))
ideal_df  = pd.read_csv(io.BytesIO(uploaded['ideal.csv']))

print(f"\n✅ train.csv  → {train_df.shape[0]} rows, columns: {list(train_df.columns)}")
print(f"✅ test.csv   → {test_df.shape[0]} rows, columns: {list(test_df.columns)}")
print(f"✅ ideal.csv  → {ideal_df.shape[0]} rows, {ideal_df.shape[1]} columns")


📂 Upload your CSV files when prompted...
   Upload: train.csv, test.csv, ideal.csv


Saving ideal.csv to ideal.csv
Saving test.csv to test.csv
Saving train.csv to train.csv

✅ train.csv  → 400 rows, columns: ['x', 'y1', 'y2', 'y3', 'y4']
✅ test.csv   → 100 rows, columns: ['x', 'y']
✅ ideal.csv  → 400 rows, 51 columns


In [4]:
# ── SQLAlchemy ORM Base ──
Base = declarative_base()


class DatasetLoadError(Exception):
    """Custom exception raised when a dataset cannot be loaded or is malformed."""
    pass


class MappingError(Exception):
    """Custom exception raised when test data cannot be mapped to ideal functions."""
    pass


In [5]:
# ── Base dataset class ──
class DatasetBase:
    """
    Base class for all dataset types.

    Provides shared loading, validation, and summary utilities.
    All specific dataset types inherit from this class.
    """

    def __init__(self, dataframe: pd.DataFrame, name: str):
        """
        Initialize DatasetBase.

        Args:
            dataframe (pd.DataFrame): The loaded dataset.
            name (str): A descriptive name for this dataset.

        Raises:
            DatasetLoadError: If the dataframe is None or empty.
        """
        if dataframe is None or dataframe.empty:
            raise DatasetLoadError(f"Dataset '{name}' is empty or could not be loaded.")
        self.df = dataframe.copy()
        self.name = name

    def summary(self):
        """Print a summary of the dataset."""
        print(f"\n📊 Dataset: {self.name}")
        print(f"   Rows: {self.df.shape[0]}, Columns: {self.df.shape[1]}")
        print(f"   Columns: {list(self.df.columns)}")

    def get_x(self) -> pd.Series:
        """Return the x column."""
        return self.df['x']


# ── Training dataset – inherits DatasetBase ──
class TrainingDataset(DatasetBase):
    """
    Represents the training dataset with 4 training functions (y1-y4).

    Inherits from DatasetBase.
    """

    def __init__(self, dataframe: pd.DataFrame):
        """
        Initialize TrainingDataset.

        Args:
            dataframe (pd.DataFrame): Training data with columns x, y1, y2, y3, y4.

        Raises:
            DatasetLoadError: If required columns are missing.
        """
        super().__init__(dataframe, "Training Data")
        required = {'x', 'y1', 'y2', 'y3', 'y4'}
        if not required.issubset(set(self.df.columns)):
            raise DatasetLoadError(f"Training data missing columns. Expected: {required}")

    def get_training_columns(self):
        """Return list of training y-column names."""
        return ['y1', 'y2', 'y3', 'y4']


# ── Ideal functions dataset – inherits DatasetBase ──
class IdealDataset(DatasetBase):
    """
    Represents the 50 ideal functions dataset.

    Inherits from DatasetBase.
    """

    def __init__(self, dataframe: pd.DataFrame):
        """
        Initialize IdealDataset.

        Args:
            dataframe (pd.DataFrame): Ideal functions data with x and y1..y50.

        Raises:
            DatasetLoadError: If x column is missing or fewer than 50 y-columns exist.
        """
        super().__init__(dataframe, "Ideal Functions")
        if 'x' not in self.df.columns:
            raise DatasetLoadError("Ideal functions dataset missing 'x' column.")
        y_cols = [c for c in self.df.columns if c != 'x']
        if len(y_cols) < 50:
            raise DatasetLoadError(f"Expected 50 ideal functions, found {len(y_cols)}.")

    def get_ideal_columns(self):
        """Return list of ideal function y-column names."""
        return [c for c in self.df.columns if c != 'x']


# ── Test dataset – inherits DatasetBase ──
class TestDataset(DatasetBase):
    """
    Represents the test dataset with x-y pairs.

    Inherits from DatasetBase.
    """

    def __init__(self, dataframe: pd.DataFrame):
        """
        Initialize TestDataset.

        Args:
            dataframe (pd.DataFrame): Test data with columns x, y.

        Raises:
            DatasetLoadError: If x or y columns are missing.
        """
        super().__init__(dataframe, "Test Data")
        if 'x' not in self.df.columns or 'y' not in self.df.columns:
            raise DatasetLoadError("Test dataset must have 'x' and 'y' columns.")


print("✅ Classes defined: DatasetBase, TrainingDataset, IdealDataset, TestDataset")


✅ Classes defined: DatasetBase, TrainingDataset, IdealDataset, TestDataset


In [6]:
# ── ORM Table: Training Data ──
class TrainingTable(Base):
    """ORM model for the training data table."""
    __tablename__ = 'training_data'

    id = Column(Integer, primary_key=True, autoincrement=True)
    x  = Column(Float, nullable=False)
    y1 = Column(Float)
    y2 = Column(Float)
    y3 = Column(Float)
    y4 = Column(Float)


# ── ORM Table: Ideal Functions ──
# We store ideal functions as dynamic columns via raw SQL (51 columns = x + y1..y50)
# The ORM model covers metadata; actual data is written via pandas to_sql

# ── ORM Table: Test Results ──
class ResultTable(Base):
    """ORM model for test mapping results table."""
    __tablename__ = 'test_results'

    id              = Column(Integer, primary_key=True, autoincrement=True)
    x_test          = Column(Float, nullable=False)
    y_test          = Column(Float, nullable=False)
    delta_y         = Column(Float)
    ideal_func_no   = Column(String)


# ── Database manager class ──
class DatabaseManager:
    """
    Manages SQLite database creation and data loading via SQLAlchemy.

    Handles all three tables: training, ideal functions, and test results.
    """

    def __init__(self, db_path: str = 'functions.db'):
        """
        Initialize the DatabaseManager.

        Args:
            db_path (str): Path to the SQLite database file.
        """
        self.engine = create_engine(f'sqlite:///{db_path}', echo=False)
        Base.metadata.create_all(self.engine)
        print(f"✅ SQLite database ready: {db_path}")

    def load_training(self, training: TrainingDataset):
        """
        Load training data into the database.

        Args:
            training (TrainingDataset): The training dataset object.
        """
        training.df.to_sql('training_data', con=self.engine,
                           if_exists='replace', index=False)
        print(f"   ✅ Training data loaded → {len(training.df)} rows")

    def load_ideal(self, ideal: IdealDataset):
        """
        Load ideal functions data into the database.

        Args:
            ideal (IdealDataset): The ideal dataset object.
        """
        ideal.df.to_sql('ideal_functions', con=self.engine,
                        if_exists='replace', index=False)
        print(f"   ✅ Ideal functions loaded → {len(ideal.df)} rows, {len(ideal.df.columns)} columns")

    def save_results(self, results_df: pd.DataFrame):
        """
        Save test mapping results into the database.

        Args:
            results_df (pd.DataFrame): DataFrame with columns:
                x_test, y_test, delta_y, ideal_func_no
        """
        results_df.to_sql('test_results', con=self.engine,
                          if_exists='replace', index=False)
        print(f"   ✅ Test results saved → {len(results_df)} rows")

    def read_table(self, table_name: str) -> pd.DataFrame:
        """
        Read a table from the database into a DataFrame.

        Args:
            table_name (str): Name of the table to read.

        Returns:
            pd.DataFrame: The table contents.
        """
        with self.engine.connect() as conn:
            return pd.read_sql(f"SELECT * FROM {table_name}", conn)


print("✅ DatabaseManager class defined")


✅ DatabaseManager class defined


In [7]:
class FunctionMatcher:
    """
    Performs the core matching logic:

    1. Finds the 4 best ideal functions from 50 using least squares.
    2. Maps test data to those 4 functions using the sqrt(2) threshold.
    """

    def __init__(self, training: TrainingDataset, ideal: IdealDataset):
        """
        Initialize FunctionMatcher.

        Args:
            training (TrainingDataset): Training dataset.
            ideal (IdealDataset): Ideal functions dataset.
        """
        self.training = training
        self.ideal    = ideal

        # Merge on x so rows align properly
        self.merged = pd.merge(training.df, ideal.df, on='x', how='inner')

        # Results populated after find_best_ideal_functions()
        self.chosen_ideals       = {}   # {train_col: ideal_col}
        self.max_deviations      = {}   # {train_col: max |y_train - y_ideal| across all x}

    def find_best_ideal_functions(self) -> dict:
        """
        For each of the 4 training functions, find the ideal function
        that minimizes the sum of squared y-deviations (least squares).

        Returns:
            dict: {training_col: best_ideal_col}
        """
        ideal_cols    = self.ideal.get_ideal_columns()
        training_cols = self.training.get_training_columns()

        print("\n🔍 Finding best ideal functions (Least Squares)...")

        for t_col in training_cols:
            best_ideal = None
            best_sse   = float('inf')  # sum of squared errors

            for i_col in ideal_cols:
                # Sum of squared deviations between training y and ideal y
                sse = ((self.merged[t_col] - self.merged[i_col]) ** 2).sum()
                if sse < best_sse:
                    best_sse   = sse
                    best_ideal = i_col

            # Calculate max absolute deviation for the chosen ideal
            max_dev = (self.merged[t_col] - self.merged[best_ideal]).abs().max()
            self.chosen_ideals[t_col]  = best_ideal
            self.max_deviations[t_col] = max_dev

            print(f"   {t_col} → {best_ideal}  (SSE={best_sse:.4f}, max_dev={max_dev:.4f})")

        return self.chosen_ideals

    def map_test_data(self, test: TestDataset) -> pd.DataFrame:
        """
        Map each test (x, y) pair to one of the 4 chosen ideal functions
        if it satisfies the sqrt(2) threshold criterion.

        Criterion:
            |y_test - y_ideal(x)| ≤ max_deviation_training * sqrt(2)

        Args:
            test (TestDataset): The test dataset.

        Returns:
            pd.DataFrame: Results with columns x_test, y_test, delta_y, ideal_func_no.

        Raises:
            MappingError: If chosen_ideals is empty (find_best_ideal_functions not called).
        """
        if not self.chosen_ideals:
            raise MappingError("Must call find_best_ideal_functions() before mapping test data.")

        results = []

        print("\n🔗 Mapping test data to chosen ideal functions...")

        for _, row in test.df.iterrows():
            x_val = row['x']
            y_val = row['y']

            best_match     = None
            best_delta     = float('inf')
            best_ideal_col = None

            for t_col, i_col in self.chosen_ideals.items():
                # Find y value of this ideal function at x_val
                ideal_row = self.ideal.df[self.ideal.df['x'] == x_val]

                if ideal_row.empty:
                    continue  # x not found in ideal — skip

                y_ideal   = ideal_row[i_col].values[0]
                delta     = abs(y_val - y_ideal)
                threshold = self.max_deviations[t_col] * math.sqrt(2)

                if delta <= threshold and delta < best_delta:
                    best_delta     = delta
                    best_match     = t_col
                    best_ideal_col = i_col

            if best_match is not None:
                results.append({
                    'x_test':        x_val,
                    'y_test':        y_val,
                    'delta_y':       round(best_delta, 6),
                    'ideal_func_no': best_ideal_col
                })
            # If no match, the point is simply not saved (per assignment spec)

        results_df = pd.DataFrame(results)
        print(f"   ✅ {len(results_df)} test points mapped out of {len(test.df)} total")
        return results_df


print("✅ FunctionMatcher class defined")



✅ FunctionMatcher class defined


In [10]:
# ── Step 1: Wrap data in dataset objects ──
print("=" * 55)
print("STEP 1 – Loading datasets into objects")
print("=" * 55)

training_ds = TrainingDataset(train_df)
ideal_ds    = IdealDataset(ideal_df)
test_ds     = TestDataset(test_df)

training_ds.summary()
ideal_ds.summary()
test_ds.summary()

# ── Step 2: Setup DB and load data ──
print("\n" + "=" * 55)
print("STEP 2 – Setting up SQLite database")
print("=" * 55)

db = DatabaseManager('functions.db')
db.load_training(training_ds)
db.load_ideal(ideal_ds)

# ── Step 3: Find best 4 ideal functions ──
print("\n" + "=" * 55)
print("STEP 3 – Finding best ideal functions (Least Squares)")
print("=" * 55)

class FunctionMatcher:
    """
    Performs the core matching logic:

    1. Finds the 4 best ideal functions from 50 using least squares.
    2. Maps test data to those 4 functions using the sqrt(2) threshold.
    """

    def __init__(self, training: TrainingDataset, ideal: IdealDataset):
        """
        Initialize FunctionMatcher.

        Args:
            training (TrainingDataset): Training dataset.
            ideal (IdealDataset): Ideal functions dataset.
        """
        self.training = training
        self.ideal    = ideal

        # Prepare training DataFrame columns with '_x' suffix
        training_df_prep = self.training.df.copy()
        training_y_cols_mapping = {col: f"{col}_x" for col in self.training.get_training_columns()}
        training_df_prep = training_df_prep.rename(columns=training_y_cols_mapping)

        # Prepare ideal DataFrame columns with '_y' suffix
        ideal_df_prep = self.ideal.df.copy()
        ideal_y_cols_mapping = {col: f"{col}_y" for col in self.ideal.get_ideal_columns()}
        ideal_df_prep = ideal_df_prep.rename(columns=ideal_y_cols_mapping)

        # Merge on x. Now all y-columns are uniquely suffixed.
        self.merged = pd.merge(training_df_prep, ideal_df_prep, on='x', how='inner')

        # Results populated after find_best_ideal_functions()
        self.chosen_ideals       = {}   # {train_col: ideal_col}
        self.max_deviations      = {}   # {train_col: max |y_train - y_ideal| across all x}

    def find_best_ideal_functions(self) -> dict:
        """
        For each of the 4 training functions, find the ideal function
        that minimizes the sum of squared y-deviations (least squares).

        Returns:
            dict: {training_col: best_ideal_col}
        """
        ideal_cols    = self.ideal.get_ideal_columns()
        training_cols = self.training.get_training_columns()

        print("\n🔍 Finding best ideal functions (Least Squares)...")

        for t_col in training_cols:
            best_ideal = None
            best_sse   = float('inf')  # sum of squared errors

            # Use the suffixed column name for the training data in the merged DataFrame
            merged_t_col = f"{t_col}_x"

            for i_col in ideal_cols:
                # Use the suffixed column name for the ideal data in the merged DataFrame
                merged_i_col = f"{i_col}_y"

                # Sum of squared deviations between training y and ideal y
                sse = ((self.merged[merged_t_col] - self.merged[merged_i_col]) ** 2).sum()
                if sse < best_sse:
                    best_sse   = sse
                    best_ideal = i_col # Store original ideal column name

            # Calculate max absolute deviation for the chosen ideal
            # Use the suffixed column name for the best ideal in the merged DataFrame
            merged_best_ideal_col = f"{best_ideal}_y"
            max_dev = (self.merged[merged_t_col] - self.merged[merged_best_ideal_col]).abs().max()
            self.chosen_ideals[t_col]  = best_ideal
            self.max_deviations[t_col] = max_dev

            print(f"   {t_col} → {best_ideal}  (SSE={best_sse:.4f}, max_dev={max_dev:.4f})")

        return self.chosen_ideals

    def map_test_data(self, test: TestDataset) -> pd.DataFrame:
        """
        Map each test (x, y) pair to one of the 4 chosen ideal functions
        if it satisfies the sqrt(2) threshold criterion.

        Criterion:
            |y_test - y_ideal(x)| ≤ max_deviation_training * sqrt(2)

        Args:
            test (TestDataset): The test dataset.

        Returns:
            pd.DataFrame: Results with columns x_test, y_test, delta_y, ideal_func_no.

        Raises:
            MappingError: If chosen_ideals is empty (find_best_ideal_functions not called).
        """
        if not self.chosen_ideals:
            raise MappingError("Must call find_best_ideal_functions() before mapping test data.")

        results = []

        print("\n🔗 Mapping test data to chosen ideal functions...")

        for _, row in test.df.iterrows():
            x_val = row['x']
            y_val = row['y']

            best_match     = None
            best_delta     = float('inf')
            best_ideal_col = None

            for t_col, i_col in self.chosen_ideals.items():
                # Find y value of this ideal function at x_val
                ideal_row = self.ideal.df[self.ideal.df['x'] == x_val]

                if ideal_row.empty:
                    continue  # x not found in ideal — skip

                y_ideal   = ideal_row[i_col].values[0]
                delta     = abs(y_val - y_ideal)
                threshold = self.max_deviations[t_col] * math.sqrt(2)

                if delta <= threshold and delta < best_delta:
                    best_delta     = delta
                    best_match     = t_col
                    best_ideal_col = i_col

            if best_match is not None:
                results.append({
                    'x_test':        x_val,
                    'y_test':        y_val,
                    'delta_y':       round(best_delta, 6),
                    'ideal_func_no': best_ideal_col
                })
            # If no match, the point is simply not saved (per assignment spec)

        results_df = pd.DataFrame(results)
        print(f"   ✅ {len(results_df)} test points mapped out of {len(test.df)} total")
        return results_df


matcher       = FunctionMatcher(training_ds, ideal_ds)
chosen        = matcher.find_best_ideal_functions()

print("\n📌 Summary of chosen ideal functions:")
for t_col, i_col in chosen.items():
    print(f"   Training {t_col}  →  Ideal function: {i_col}")

# ── Step 4: Map test data ──
print("\n" + "=" * 55)
print("STEP 4 – Mapping test data")
print("=" * 55)

results_df = matcher.map_test_data(test_ds)
print(results_df.head(10))

# ── Step 5: Save results to DB ──
print("\n" + "=" * 55)
print("STEP 5 – Saving results to database")
print("=" * 55)

db.save_results(results_df)

# Verify by reading back
print("\n📋 Verification – reading test_results from DB:")
print(db.read_table('test_results').head())

STEP 1 – Loading datasets into objects

📊 Dataset: Training Data
   Rows: 400, Columns: 5
   Columns: ['x', 'y1', 'y2', 'y3', 'y4']

📊 Dataset: Ideal Functions
   Rows: 400, Columns: 51
   Columns: ['x', 'y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10', 'y11', 'y12', 'y13', 'y14', 'y15', 'y16', 'y17', 'y18', 'y19', 'y20', 'y21', 'y22', 'y23', 'y24', 'y25', 'y26', 'y27', 'y28', 'y29', 'y30', 'y31', 'y32', 'y33', 'y34', 'y35', 'y36', 'y37', 'y38', 'y39', 'y40', 'y41', 'y42', 'y43', 'y44', 'y45', 'y46', 'y47', 'y48', 'y49', 'y50']

📊 Dataset: Test Data
   Rows: 100, Columns: 2
   Columns: ['x', 'y']

STEP 2 – Setting up SQLite database
✅ SQLite database ready: functions.db
   ✅ Training data loaded → 400 rows
   ✅ Ideal functions loaded → 400 rows, 51 columns

STEP 3 – Finding best ideal functions (Least Squares)

🔍 Finding best ideal functions (Least Squares)...
   y1 → y13  (SSE=34.0807, max_dev=0.4992)
   y2 → y24  (SSE=33.4518, max_dev=0.4990)
   y3 → y36  (SSE=35.5727, max

In [11]:
class Visualizer:
    """
    Handles all Bokeh visualizations for training, ideal, and test data.
    """

    COLORS = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3',
              '#ff7f00', '#a65628', '#f781bf', '#999999']

    def __init__(self, training: TrainingDataset, ideal: IdealDataset,
                 results: pd.DataFrame, chosen: dict):
        """
        Initialize Visualizer.

        Args:
            training (TrainingDataset): Training data.
            ideal (IdealDataset): Ideal functions data.
            results (pd.DataFrame): Mapping results.
            chosen (dict): {train_col: ideal_col} mapping.
        """
        self.training = training
        self.ideal    = ideal
        self.results  = results
        self.chosen   = chosen

    def plot_training_vs_ideal(self):
        """
        Plot each training function alongside its matched ideal function.
        Creates a 2x2 grid of plots.
        """
        plots = []
        x_vals = self.training.df['x'].values

        for idx, (t_col, i_col) in enumerate(self.chosen.items()):
            p = figure(
                title=f"Training {t_col}  vs  Ideal {i_col}",
                width=500, height=350,
                x_axis_label='x',
                y_axis_label='y',
                tools="pan,wheel_zoom,box_zoom,reset,hover"
            )

            # Training data as circles
            p.circle(
                x=x_vals,
                y=self.training.df[t_col].values,
                size=6,
                color=self.COLORS[idx * 2],
                legend_label=f"Training {t_col}",
                alpha=0.7
            )

            # Ideal function as line
            ideal_x = self.ideal.df['x'].values
            ideal_y = self.ideal.df[i_col].values

            p.line(
                x=ideal_x,
                y=ideal_y,
                line_width=2,
                color=self.COLORS[idx * 2 + 1],
                legend_label=f"Ideal {i_col}",
                alpha=0.9
            )

            p.legend.location = "top_left"
            p.legend.click_policy = "hide"
            plots.append(p)

        grid = gridplot([plots[:2], plots[2:]])
        show(grid)

    def plot_test_results(self):
        """
        Plot test data points colored by which ideal function they mapped to.
        Unmapped points are shown in grey.
        """
        p = figure(
            title="Test Data – Mapped to Ideal Functions",
            width=800, height=450,
            x_axis_label='x',
            y_axis_label='y',
            tools="pan,wheel_zoom,box_zoom,reset,hover"
        )

        # All test data in grey first
        p.circle(
            x=self.training.df['x'].values,  # background ref
            y=[0] * len(self.training.df),
            size=0  # invisible — just for sizing
        )

        ideal_cols = list(self.chosen.values())
        for idx, i_col in enumerate(ideal_cols):
            subset = self.results[self.results['ideal_func_no'] == i_col]
            if subset.empty:
                continue
            p.circle(
                x=subset['x_test'].values,
                y=subset['y_test'].values,
                size=8,
                color=self.COLORS[idx],
                legend_label=f"Mapped → {i_col}",
                alpha=0.8
            )

        p.legend.location = "top_left"
        p.legend.click_policy = "hide"
        show(p)

    def plot_deviations(self):
        """
        Bar chart showing the delta_y deviations for each mapped test point.
        """
        if self.results.empty:
            print("⚠️  No results to plot deviations for.")
            return

        p = figure(
            title="Deviation (Delta Y) for Mapped Test Points",
            width=800, height=350,
            x_axis_label='Test Point Index',
            y_axis_label='|Delta Y|',
            tools="pan,wheel_zoom,reset,hover"
        )

        indices = list(range(len(self.results)))
        p.vbar(
            x=indices,
            top=self.results['delta_y'].values,
            width=0.6,
            color='#2196F3',
            alpha=0.8
        )

        show(p)


# ── Run visualizations ──
print("=" * 55)
print("STEP 6 – Visualizations")
print("=" * 55)

viz = Visualizer(training_ds, ideal_ds, results_df, chosen)

print("\n📈 Plot 1: Training vs Ideal Functions")
viz.plot_training_vs_ideal()

print("\n📈 Plot 2: Test Data Mapping")
viz.plot_test_results()

print("\n📈 Plot 3: Deviations")
viz.plot_deviations()



STEP 6 – Visualizations

📈 Plot 1: Training vs Ideal Functions



📈 Plot 2: Test Data Mapping



📈 Plot 3: Deviations


In [12]:
class TestDatasetLoading(unittest.TestCase):
    """Unit tests for dataset loading and validation."""

    def setUp(self):
        """Create minimal mock DataFrames for testing."""
        self.valid_train = pd.DataFrame({
            'x':  [-1.0, 0.0, 1.0],
            'y1': [1.0, 0.0, 1.0],
            'y2': [2.0, 0.0, 2.0],
            'y3': [3.0, 0.0, 3.0],
            'y4': [4.0, 0.0, 4.0],
        })
        self.valid_test = pd.DataFrame({'x': [0.5], 'y': [1.2]})

        # Ideal: x + 50 y columns
        ideal_data = {'x': [-1.0, 0.0, 1.0]}
        for i in range(1, 51):
            ideal_data[f'y{i}'] = [float(i), 0.0, float(i)]
        self.valid_ideal = pd.DataFrame(ideal_data)

    def test_training_dataset_loads_ok(self):
        """TrainingDataset should load without errors given correct data."""
        ds = TrainingDataset(self.valid_train)
        self.assertEqual(len(ds.df), 3)

    def test_training_dataset_missing_column_raises(self):
        """TrainingDataset should raise DatasetLoadError if column is missing."""
        bad_df = self.valid_train.drop(columns=['y4'])
        with self.assertRaises(DatasetLoadError):
            TrainingDataset(bad_df)

    def test_empty_dataframe_raises(self):
        """Empty DataFrame should raise DatasetLoadError."""
        with self.assertRaises(DatasetLoadError):
            TrainingDataset(pd.DataFrame())

    def test_ideal_dataset_loads_ok(self):
        """IdealDataset should load without errors given 50 y columns."""
        ds = IdealDataset(self.valid_ideal)
        self.assertEqual(len(ds.get_ideal_columns()), 50)

    def test_ideal_dataset_missing_x_raises(self):
        """IdealDataset should raise DatasetLoadError if x column is missing."""
        bad = self.valid_ideal.drop(columns=['x'])
        with self.assertRaises(DatasetLoadError):
            IdealDataset(bad)

    def test_test_dataset_loads_ok(self):
        """TestDataset should load correctly with x and y columns."""
        ds = TestDataset(self.valid_test)
        self.assertIn('x', ds.df.columns)
        self.assertIn('y', ds.df.columns)

    def test_test_dataset_missing_y_raises(self):
        """TestDataset should raise DatasetLoadError if y column is missing."""
        bad = pd.DataFrame({'x': [1.0]})
        with self.assertRaises(DatasetLoadError):
            TestDataset(bad)


class TestFunctionMatcher(unittest.TestCase):
    """Unit tests for the FunctionMatcher core logic."""

    def setUp(self):
        """Set up simple predictable datasets for testing matching logic."""
        x_vals = pd.Series([-1.0, 0.0, 1.0])

        train_data = {
            'x':  x_vals,
            'y1': x_vals * 2,       # y = 2x
            'y2': x_vals ** 2,      # y = x^2
            'y3': x_vals * -1,      # y = -x
            'y4': x_vals * 3 + 1,   # y = 3x + 1
        }
        self.train_df = pd.DataFrame(train_data)

        ideal_data = {'x': x_vals}
        # y1..y10: various functions; y1 = 2x exactly (perfect match for training y1)
        for i in range(1, 51):
            ideal_data[f'y{i}'] = x_vals * i
        # Override y1 to perfectly match training y1
        ideal_data['y1'] = x_vals * 2
        self.ideal_df = pd.DataFrame(ideal_data)

        self.training = TrainingDataset(self.train_df)
        self.ideal    = IdealDataset(self.ideal_df)

    def test_find_best_returns_4_functions(self):
        """find_best_ideal_functions should return exactly 4 matches."""
        matcher = FunctionMatcher(self.training, self.ideal)
        result  = matcher.find_best_ideal_functions()
        self.assertEqual(len(result), 4)

    def test_mapping_raises_without_finding_first(self):
        """map_test_data should raise MappingError if called before finding ideals."""
        matcher  = FunctionMatcher(self.training, self.ideal)
        test_df  = pd.DataFrame({'x': [0.0], 'y': [0.0]})
        test_ds  = TestDataset(test_df)
        with self.assertRaises(MappingError):
            matcher.map_test_data(test_ds)

    def test_mapping_returns_dataframe(self):
        """map_test_data should return a DataFrame with expected columns."""
        matcher = FunctionMatcher(self.training, self.ideal)
        matcher.find_best_ideal_functions()
        test_ds = TestDataset(pd.DataFrame({'x': [0.0], 'y': [0.0]}))
        result  = matcher.map_test_data(test_ds)
        self.assertIsInstance(result, pd.DataFrame)
        for col in ['x_test', 'y_test', 'delta_y', 'ideal_func_no']:
            self.assertIn(col, result.columns)


# ── Run all unit tests ──
print("=" * 55)
print("STEP 7 – Running Unit Tests")
print("=" * 55)

loader = unittest.TestLoader()
suite  = unittest.TestSuite()
suite.addTests(loader.loadTestsFromTestCase(TestDatasetLoading))
suite.addTests(loader.loadTestsFromTestCase(TestFunctionMatcher))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)


test_empty_dataframe_raises (__main__.TestDatasetLoading.test_empty_dataframe_raises)
Empty DataFrame should raise DatasetLoadError. ... ok
test_ideal_dataset_loads_ok (__main__.TestDatasetLoading.test_ideal_dataset_loads_ok)
IdealDataset should load without errors given 50 y columns. ... ok
test_ideal_dataset_missing_x_raises (__main__.TestDatasetLoading.test_ideal_dataset_missing_x_raises)
IdealDataset should raise DatasetLoadError if x column is missing. ... ok
test_test_dataset_loads_ok (__main__.TestDatasetLoading.test_test_dataset_loads_ok)
TestDataset should load correctly with x and y columns. ... ok
test_test_dataset_missing_y_raises (__main__.TestDatasetLoading.test_test_dataset_missing_y_raises)
TestDataset should raise DatasetLoadError if y column is missing. ... ok
test_training_dataset_loads_ok (__main__.TestDatasetLoading.test_training_dataset_loads_ok)
TrainingDataset should load without errors given correct data. ... ok
test_training_dataset_missing_column_raises (__ma

STEP 7 – Running Unit Tests

🔍 Finding best ideal functions (Least Squares)...
   y1 → y1  (SSE=0.0000, max_dev=0.0000)
   y2 → y1  (SSE=10.0000, max_dev=3.0000)
   y3 → y1  (SSE=18.0000, max_dev=3.0000)
   y4 → y3  (SSE=3.0000, max_dev=1.0000)


ok
test_mapping_returns_dataframe (__main__.TestFunctionMatcher.test_mapping_returns_dataframe)
map_test_data should return a DataFrame with expected columns. ... ok

----------------------------------------------------------------------
Ran 10 tests in 0.322s

OK



🔍 Finding best ideal functions (Least Squares)...
   y1 → y1  (SSE=0.0000, max_dev=0.0000)
   y2 → y1  (SSE=10.0000, max_dev=3.0000)
   y3 → y1  (SSE=18.0000, max_dev=3.0000)
   y4 → y3  (SSE=3.0000, max_dev=1.0000)

🔗 Mapping test data to chosen ideal functions...
   ✅ 1 test points mapped out of 1 total


<unittest.runner.TextTestResult run=10 errors=0 failures=0>

In [ ]:
git_commands = """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SECTION 1.3 – Git Commands
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

── Clone the develop branch to your local PC ──

    git clone --branch develop https://github.com/your-org/your-repo.git
    cd your-repo

── Create a new feature branch for your work ──

    git checkout -b feature/add-new-function

── (Make your changes to the code) ──

── Stage all changed files ──

    git add .

── Commit with a descriptive message ──

    git commit -m "feat: add new function for XYZ"

── Push your feature branch to remote ──

    git push origin feature/add-new-function

── Then on GitHub/GitLab: ──
    → Open a Pull Request from feature/add-new-function → develop
    → Team reviews the code
    → After approval, the PR is merged into develop

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

print(git_commands)

print("\n🎉 ALL DONE! Full solution complete.")
print("   Database file: functions.db")
print("   Tables: training_data, ideal_functions, test_results")

